# 📊 Session 3: Interactive Statistical Distributions & Summary Metrics
Welcome to **Session 3 (Interactive Edition)**! This notebook features fully interactive widgets (`ipywidgets`) for every concept. Use the sliders and dropdowns in each cell to explore how parameters dynamically impact statistical distributions and summary metrics.

### 📚 Table of Contents
1. [Interactive Normal (Gaussian) Distribution](#1.-Normal-(Gaussian)-Distribution)
2. [Interactive Mean, Variance, and Standard Deviation (Bessel's Correction)](#2.-Mean,-Variance,-and-Standard-Deviation)
3. [Interactive Mean Absolute Deviation (MAD)](#3.-Mean-Absolute-Deviation-(MAD))
4. [Interactive Coefficient of Variation (CV)](#4.-Coefficient-of-Variation-(CV))
5. [Interactive Range](#5.-Range)
6. [Interactive Central Limit Theorem (CLT)](#6.-Central-Limit-Theorem-(CLT))
7. [Interactive Median, Quantiles, and Interquartile Range (IQR)](#7.-Median,-Quantiles,-and-Interquartile-Range-(IQR))
8. [Interactive Mode (Unimodal, Bimodal, Trimodal, Multimodal)](#8.-Mode)
9. [Interactive Skewness](#9.-Skewness)
10. [Interactive Kurtosis](#10.-Kurtosis)


In [1]:
# Setup and Imports
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
from IPython.display import display, HTML, Markdown, clear_output

# Styling setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 16

print("Libraries imported! Move sliders below to interactively explore each distribution.")


Libraries imported! Move sliders below to interactively explore each distribution.


## 1. Normal (Gaussian) Distribution
- Adjust **Mean ($\mu$)** to shift the center of the distribution.
- Adjust **Std Dev ($\sigma$)** to expand or shrink the spread.
- Select the **Shade Region** dropdown to visualize empirical probability bounds ($\pm 1\sigma, \pm 2\sigma, \pm 3\sigma$).


In [2]:
# Interactive Normal Distribution
def plot_interactive_normal(mu=0.0, sigma=1.0, sd_shade=1):
    plt.close('all')
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.linspace(mu - 4*sigma, mu + 4*sigma, 1000)
    y = stats.norm.pdf(x, mu, sigma)
    
    ax.plot(x, y, color='#2c3e50', lw=2.5, label=f'N(μ={mu}, σ={sigma})')
    
    # Shade requested SD range
    x_shade = np.linspace(mu - sd_shade*sigma, mu + sd_shade*sigma, 500)
    y_shade = stats.norm.pdf(x_shade, mu, sigma)
    prob_area = stats.norm.cdf(mu + sd_shade*sigma, mu, sigma) - stats.norm.cdf(mu - sd_shade*sigma, mu, sigma)
    
    ax.fill_between(x_shade, y_shade, color='#3498db', alpha=0.5, label=f'Area = {prob_area*100:.2f}% (±{sd_shade}σ)')
    ax.axvline(mu, color='#d62728', linestyle='--', lw=2, label=f'Mean (μ = {mu})')
    
    ax.set_title(f'Interactive Normal Distribution: Mean = {mu}, SD = {sigma}')
    ax.set_xlabel('Value (x)')
    ax.set_ylabel('Probability Density')
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

interact(
    plot_interactive_normal,
    mu=FloatSlider(min=-10.0, max=10.0, step=0.5, value=0.0, description='Mean (μ):'),
    sigma=FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='Std Dev (σ):'),
    sd_shade=Dropdown(options=[('±1σ (68.27%)', 1), ('±2σ (95.45%)', 2), ('±3σ (99.73%)', 3)], value=1, description='Shade Region:')
);


interactive(children=(FloatSlider(value=0.0, description='Mean (μ):', max=10.0, min=-10.0, step=0.5), FloatSli…

## 2. Mean, Variance, and Standard Deviation
- Explore **Bessel's Correction** ($n-1$ vs $n$) by varying sample size $n$.
- Observe how sample mean $\bar{x}$ converges to true population mean $\mu$ as sample size increases.


In [3]:
# Interactive Bessel's Correction & Sample Variance Simulation
def plot_interactive_bessel(pop_mean=50.0, pop_sd=10.0, sample_size=20, trials=500):
    plt.close('all')
    np.random.seed(42)
    pop = np.random.normal(loc=pop_mean, scale=pop_sd, size=10000)
    true_var = pop_sd ** 2
    
    samples = np.random.choice(pop, size=(trials, sample_size))
    var_n = np.mean(np.var(samples, axis=1, ddof=0))
    var_n_1 = np.mean(np.var(samples, axis=1, ddof=1))
    sample_means = np.mean(samples, axis=1)
    
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart comparing estimates vs true variance
    ax[0].bar(['Divided by n\n(Biased)', "Divided by (n-1)\n(Bessel's Unbiased)", 'True Variance\n(σ²)'], 
               [var_n, var_n_1, true_var], 
               color=['#d62728', '#2ca02c', '#2c3e50'], alpha=0.8)
    ax[0].axhline(true_var, color='#2c3e50', linestyle='--', lw=2)
    ax[0].set_title(f'Variance Estimation Comparison (n = {sample_size})\nBiased: {var_n:.2f} | Unbiased: {var_n_1:.2f} | True: {true_var:.2f}')
    ax[0].set_ylabel('Variance')
    
    # Distribution of sample means
    sns.histplot(sample_means, kde=True, ax=ax[1], color='#3498db', stat='density')
    ax[1].axvline(pop_mean, color='#d62728', linestyle='--', lw=2, label=f'True μ ({pop_mean})')
    ax[1].axvline(np.mean(sample_means), color='#2ca02c', linestyle=':', lw=2, label=f'Avg x̄ ({np.mean(sample_means):.2f})')
    ax[1].set_title(f'Sampling Distribution of Mean x̄ (Trials = {trials})')
    ax[1].set_xlabel('Sample Mean x̄')
    ax[1].legend()
    
    plt.tight_layout()
    plt.show()

interact(
    plot_interactive_bessel,
    pop_mean=FloatSlider(min=10.0, max=100.0, step=5.0, value=50.0, description='Pop Mean (μ):'),
    pop_sd=FloatSlider(min=1.0, max=30.0, step=1.0, value=10.0, description='Pop SD (σ):'),
    sample_size=IntSlider(min=5, max=200, step=5, value=20, description='Sample Size (n):'),
    trials=IntSlider(min=100, max=2000, step=100, value=500, description='Trials:')
);


interactive(children=(FloatSlider(value=50.0, description='Pop Mean (μ):', min=10.0, step=5.0), FloatSlider(va…

In [16]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

def plot_convergence(true_mean=50, true_std=10, n_max=500, seed=42):
    np.random.seed(seed)
    population_samples = np.random.normal(true_mean, true_std, n_max)

    sample_sizes = np.arange(2, n_max + 1)
    naive_means = []       # standard sample mean: sum / n
    corrected_means = []   # Bessel-style: sum / (n - 1)

    for n in sample_sizes:
        data = population_samples[:n]
        total = data.sum()
        naive_means.append(total / n)
        corrected_means.append(total / (n - 1))

    plt.figure(figsize=(10, 6))
    plt.plot(sample_sizes, naive_means, label="Estimated Mean (÷n)", color="tab:blue")
    plt.plot(sample_sizes, corrected_means, label="Bessel-Corrected Mean (÷(n-1))", color="tab:orange")
    plt.axhline(true_mean, color="black", linestyle="--", label="True Mean")

    plt.xlabel("Sample Size (n)")
    plt.ylabel("Mean Estimate")
    plt.title(f"Convergence of Mean Estimates (true mean={true_mean}, std={true_std})")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.ylim(true_mean - true_std, true_mean + true_std)
    plt.tight_layout()
    plt.show()

interact(
    plot_convergence,
    true_mean=FloatSlider(min=0, max=100, step=1, value=50),
    true_std=FloatSlider(min=1, max=30, step=1, value=10),
    n_max=IntSlider(min=10, max=2000, step=10, value=500),
    seed=IntSlider(min=0, max=100, step=1, value=42),
)

interactive(children=(FloatSlider(value=50.0, description='true_mean', step=1.0), FloatSlider(value=10.0, desc…

<function __main__.plot_convergence(true_mean=50, true_std=10, n_max=500, seed=42)>

## 3. Mean Absolute Deviation (MAD)
- Drag individual data point sliders to see linear deviations $|x - \bar{x}|$ vs squared deviations $(x - \bar{x})^2$.
- Watch how extreme outlier values disproportionately affect Standard Deviation compared to MAD.


In [4]:
# Interactive MAD vs Standard Deviation
def plot_interactive_mad(outlier_val=45.0, spread=5.0):
    plt.close('all')
    np.random.seed(42)
    base_pts = np.round(np.random.normal(20, spread, 5), 1)
    pts = np.append(base_pts, outlier_val)
    m_val = np.mean(pts)
    abs_d = np.abs(pts - m_val)
    sq_d = (pts - m_val)**2
    
    mad = np.mean(abs_d)
    sd = np.std(pts, ddof=1)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    y_pos = np.arange(len(pts))
    
    ax.scatter(pts, y_pos, color='#2c3e50', s=150, zorder=5, label='Data Points')
    ax.axvline(m_val, color='#d62728', linestyle='--', lw=2, label=f'Mean (x̄ = {m_val:.1f})')
    
    for idx, (x_val, y) in enumerate(zip(pts, y_pos)):
        ax.plot([m_val, x_val], [y, y], color='#e67e22', lw=3, zorder=3)
        ax.text((m_val + x_val)/2, y + 0.15, f'|x-x̄|={abs_d[idx]:.1f} | (x-x̄)²={sq_d[idx]:.1f}', 
                ha='center', fontsize=9, bbox=dict(boxstyle='round,pad=0.2', facecolor='yellow', alpha=0.3))
        
    ax.set_yticks(y_pos)
    labels = [f'P{i+1} ({v})' for i, v in enumerate(pts[:-1])] + [f'Outlier ({pts[-1]})']
    ax.set_yticklabels(labels)
    ax.set_title(f'Interactive MAD vs SD\nCalculated MAD = {mad:.2f} | Sample SD (s) = {sd:.2f}')
    ax.set_xlabel('Value')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

interact(
    plot_interactive_mad,
    outlier_val=FloatSlider(min=20.0, max=80.0, step=2.0, value=45.0, description='Outlier Value:'),
    spread=FloatSlider(min=1.0, max=15.0, step=1.0, value=5.0, description='Data Spread:')
);


interactive(children=(FloatSlider(value=45.0, description='Outlier Value:', max=80.0, min=20.0, step=2.0), Flo…

## 4. Coefficient of Variation (CV)
- Adjust the mean and SD of Dataset A vs Dataset B.
- See why absolute SD can be misleading without comparing relative spread ($\text{CV} = \frac{\text{SD}}{\text{Mean}} \times 100\%$).


In [5]:
# Interactive Coefficient of Variation (CV)
def plot_interactive_cv(mean_A=10.0, sd_A=2.0, mean_B=1000.0, sd_B=200.0):
    plt.close('all')
    cv_A = (sd_A / mean_A) * 100
    cv_B = (sd_B / mean_B) * 100
    
    np.random.seed(42)
    data_A = np.random.normal(loc=mean_A, scale=sd_A, size=1000)
    data_B = np.random.normal(loc=mean_B, scale=sd_B, size=1000)
    
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.kdeplot(data_A, ax=ax[0], color='#2980b9', fill=True, alpha=0.4)
    ax[0].set_title(f'Dataset A: Mean = {mean_A}, SD = {sd_A}\nRelative CV = {cv_A:.2f}%')
    ax[0].set_xlabel('Value')
    
    sns.kdeplot(data_B, ax=ax[1], color='#27ae60', fill=True, alpha=0.4)
    ax[1].set_title(f'Dataset B: Mean = {mean_B}, SD = {sd_B}\nRelative CV = {cv_B:.2f}%')
    ax[1].set_xlabel('Value')
    
    plt.tight_layout()
    plt.show()

interact(
    plot_interactive_cv,
    mean_A=FloatSlider(min=5.0, max=50.0, step=1.0, value=10.0, description='Mean A:'),
    sd_A=FloatSlider(min=0.5, max=10.0, step=0.5, value=2.0, description='SD A:'),
    mean_B=FloatSlider(min=100.0, max=5000.0, step=100.0, value=1000.0, description='Mean B:'),
    sd_B=FloatSlider(min=10.0, max=500.0, step=10.0, value=200.0, description='SD B:')
);


interactive(children=(FloatSlider(value=10.0, description='Mean A:', max=50.0, min=5.0, step=1.0), FloatSlider…

## 5. Range
- Change **Min Value** and **Max Value** to interactively observe the range span.


In [6]:
# Interactive Range
def plot_interactive_range(center=50, spread_range=40, num_points=15):
    plt.close('all')
    min_val = center - spread_range // 2
    max_val = center + spread_range // 2
    np.random.seed(42)
    middle_pts = np.random.uniform(min_val, max_val, num_points - 2)
    pts = np.sort(np.concatenate([[min_val, max_val], middle_pts]))
    r_val = max_val - min_val
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.scatter(pts, np.zeros_like(pts), color='#1f77b4', s=120, zorder=4, label='Data Points')
    ax.axvline(min_val, color='#d62728', linestyle='--', lw=2, label=f'Min ({min_val})')
    ax.axvline(max_val, color='#2ca02c', linestyle='--', lw=2, label=f'Max ({max_val})')
    ax.annotate('', xy=(min_val, 0.2), xytext=(max_val, 0.2), arrowprops=dict(arrowstyle='<->', lw=2.5, color='#2c3e50'))
    ax.text((min_val + max_val)/2, 0.25, f'Range = Max - Min = {r_val}', ha='center', fontsize=12, fontweight='bold', bbox=dict(boxstyle='round', facecolor='white', edgecolor='#2c3e50'))
    
    ax.set_ylim(-0.5, 0.8)
    ax.set_yticks([])
    ax.set_title(f'Interactive Range Span (Min = {min_val}, Max = {max_val})')
    ax.set_xlabel('Value')
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

interact(
    plot_interactive_range,
    center=IntSlider(min=20, max=100, step=5, value=50, description='Center Value:'),
    spread_range=IntSlider(min=10, max=80, step=5, value=40, description='Span / Range:'),
    num_points=IntSlider(min=5, max=50, step=5, value=15, description='Points:')
);


interactive(children=(IntSlider(value=50, description='Center Value:', min=20, step=5), IntSlider(value=40, de…

## 6. Central Limit Theorem (CLT)
- Select different parent population distributions (Exponential, Uniform, Bimodal).
- Increase **Sample Size ($n$)** from $1$ to $100$ and watch the distribution of sample means transform into a Gaussian curve!


In [15]:
# 1. Add this magic command to tell Jupyter to render plots directly in the notebook
%matplotlib inline

# 2. Ensure all required libraries are imported
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, Dropdown, IntSlider

# Interactive Central Limit Theorem (CLT)
def plot_interactive_clt(dist_type='Exponential (Skewed)', n=30, num_samples=2000):
    # Clear previous figures to prevent memory leaks and blank overlapping plots
    plt.close('all')
    
    np.random.seed(42)
    if 'Exponential' in dist_type:
        pop = np.random.exponential(scale=2.0, size=50000)
    elif 'Uniform' in dist_type:
        pop = np.random.uniform(low=0, high=10, size=50000)
    else:
        pop = np.concatenate([np.random.normal(5, 1, 25000), np.random.normal(15, 1, 25000)])
        
    sample_means = [np.mean(np.random.choice(pop, size=n)) for _ in range(num_samples)]
    
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    
    # Population
    sns.histplot(pop, kde=True, ax=ax[0], color='#8e44ad', stat='density')
    ax[0].set_title(f'Parent Population ({dist_type})')
    ax[0].set_xlabel('Value')
    
    # Sample Means
    sns.histplot(sample_means, kde=True, ax=ax[1], color='#2ca02c' if n >= 30 else '#1f77b4', stat='density')
    ax[1].set_title(f'Sampling Distribution of Means (n = {n})\n{"Approx. Normal (CLT Holds!)" if n >= 30 else "Not yet normal (n < 30)"}')
    ax[1].set_xlabel('Sample Mean x̄')
    
    plt.tight_layout()
    plt.show()

# 3. Render the interactive widget
interact(
    plot_interactive_clt,
    dist_type=Dropdown(options=['Exponential (Skewed)', 'Uniform (Flat)', 'Bimodal (Two Peaks)'], value='Exponential (Skewed)', description='Population:'),
    n=IntSlider(min=1, max=100, step=1, value=30, description='Sample Size (n):'),
    num_samples=IntSlider(min=500, max=5000, step=500, value=2000, description='Num Samples:')
);

interactive(children=(Dropdown(description='Population:', options=('Exponential (Skewed)', 'Uniform (Flat)', '…

## 7. Median, Quantiles, and Interquartile Range (IQR)
- Adjust the **Fence Multiplier** (standard is $1.5 \times \text{IQR}$).
- Add extreme outliers and observe how outlier detection fences dynamically update.


In [8]:
# Interactive IQR & Outliers Boxplot
def plot_interactive_iqr(fence_multiplier=1.5, outlier_count=4):
    plt.close('all')
    np.random.seed(10)
    clean = np.random.normal(loc=50, scale=10, size=200)
    if outlier_count > 0:
        outs = np.linspace(85, 120, outlier_count)
        data = np.concatenate([clean, outs])
    else:
        data = clean
        
    q1 = np.percentile(data, 25)
    med = np.median(data)
    q3 = np.percentile(data, 75)
    iqr = q3 - q1
    lower_f = q1 - fence_multiplier * iqr
    upper_f = q3 + fence_multiplier * iqr
    
    fig, ax = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={'height_ratios': [2, 1]})
    
    sns.histplot(data, kde=True, ax=ax[0], color='#34495e', bins=30, alpha=0.5)
    ax[0].axvline(q1, color='#1f77b4', linestyle='--', lw=2, label=f'Q1: {q1:.1f}')
    ax[0].axvline(med, color='#2ca02c', linestyle='-', lw=2.5, label=f'Median: {med:.1f}')
    ax[0].axvline(q3, color='#1f77b4', linestyle='--', lw=2, label=f'Q3: {q3:.1f}')
    ax[0].axvline(lower_f, color='#d62728', linestyle=':', lw=2, label=f'Lower Fence ({fence_multiplier:.1f}×IQR): {lower_f:.1f}')
    ax[0].axvline(upper_f, color='#d62728', linestyle=':', lw=2, label=f'Upper Fence ({fence_multiplier:.1f}×IQR): {upper_f:.1f}')
    ax[0].set_title(f'Interactive IQR & Outlier Fences (Fence Multiplier = {fence_multiplier:.1f}×IQR)')
    ax[0].legend()
    
    sns.boxplot(x=data, ax=ax[1], color='#3498db', flierprops=dict(marker='o', markerfacecolor='r', markersize=8))
    ax[1].set_title(f'Boxplot (IQR = {iqr:.2f})')
    ax[1].set_xlabel('Value')
    
    plt.tight_layout()
    plt.show()

interact(
    plot_interactive_iqr,
    fence_multiplier=FloatSlider(min=0.5, max=3.0, step=0.1, value=1.5, description='Fence Multiplier:'),
    outlier_count=IntSlider(min=0, max=10, step=1, value=4, description='Outliers Count:')
);


interactive(children=(FloatSlider(value=1.5, description='Fence Multiplier:', max=3.0, min=0.5), IntSlider(val…

## 8. Mode
- Switch between **Unimodal**, **Bimodal**, **Trimodal**, and **Multimodal** distributions.
- Adjust the **Peak Gap** slider to separate or merge modal peaks.


In [9]:
# Interactive Mode
def plot_interactive_mode(mode_type='Bimodal', separation=15.0):
    plt.close('all')
    np.random.seed(42)
    if mode_type == 'Unimodal':
        data = np.random.normal(loc=20, scale=4, size=2000)
    elif mode_type == 'Bimodal':
        data = np.concatenate([np.random.normal(20, 3, 1000), np.random.normal(20 + separation, 3, 1000)])
    elif mode_type == 'Trimodal':
        data = np.concatenate([np.random.normal(20, 2, 800), np.random.normal(20 + separation, 2, 800), np.random.normal(20 + 2*separation, 2, 800)])
    else:
        data = np.concatenate([np.random.normal(20 + i*separation, 1.5, 500) for i in range(4)])
        
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.kdeplot(data, ax=ax, color='#8e44ad', fill=True, alpha=0.4)
    ax.set_title(f'Interactive Mode Visualization: {mode_type} (Peak Separation = {separation})')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    plt.tight_layout()
    plt.show()

interact(
    plot_interactive_mode,
    mode_type=Dropdown(options=['Unimodal', 'Bimodal', 'Trimodal', 'Multimodal'], value='Bimodal', description='Mode Type:'),
    separation=FloatSlider(min=5.0, max=30.0, step=1.0, value=15.0, description='Peak Gap:')
);


interactive(children=(Dropdown(description='Mode Type:', index=1, options=('Unimodal', 'Bimodal', 'Trimodal', …

## 9. Skewness
- Adjust **Skewness ($\alpha$)** from negative to positive.
- Watch how **Mean**, **Median**, and **Mode** line up when symmetrical, and split apart when skewed!


In [10]:
# Interactive Skewness
def plot_interactive_skewness(skew_alpha=4.0):
    plt.close('all')
    np.random.seed(42)
    data = stats.skewnorm.rvs(skew_alpha, loc=50, scale=10, size=5000)
    
    mean_v = np.mean(data)
    median_v = np.median(data)
    mode_v = stats.mode(np.round(data, 1), keepdims=True).mode[0]
    calc_skew = stats.skew(data)
    pearson_skew = (mean_v - mode_v) / np.std(data, ddof=1)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.kdeplot(data, ax=ax, color='#e67e22', fill=True, alpha=0.4)
    
    ax.axvline(mode_v, color='#2ca02c', linestyle=':', lw=2.5, label=f'Mode ({mode_v:.1f})')
    ax.axvline(median_v, color='#1f77b4', linestyle='--', lw=2.5, label=f'Median ({median_v:.1f})')
    ax.axvline(mean_v, color='#d62728', linestyle='-', lw=2.5, label=f'Mean ({mean_v:.1f})')
    
    skew_label = "Positive (Right) Skew" if skew_alpha > 0.5 else ("Negative (Left) Skew" if skew_alpha < -0.5 else "Symmetrical")
    ax.set_title(f'Interactive Skewness ({skew_label})\nSample Skewness = {calc_skew:.2f} | Pearson\'s Coeff = {pearson_skew:.2f}')
    ax.set_xlabel('Value')
    ax.legend()
    plt.tight_layout()
    plt.show()

interact(
    plot_interactive_skewness,
    skew_alpha=FloatSlider(min=-8.0, max=8.0, step=0.5, value=4.0, description='Skewness α:')
);


interactive(children=(FloatSlider(value=4.0, description='Skewness α:', max=8.0, min=-8.0, step=0.5), Output()…

## 10. Kurtosis
- Select between **Mesokurtic (Normal)**, **Leptokurtic (Laplace/Heavy Tails)**, and **Platykurtic (Uniform/Light Tails)**.
- Modify scale width to see how tail density compares to standard normal distribution.


In [11]:
# Interactive Kurtosis
def plot_interactive_kurtosis(dist_choice='normal', scale_param=1.0):
    plt.close('all')
    x = np.linspace(-6, 6, 1000)
    
    if dist_choice == 'normal':
        y = stats.norm.pdf(x, 0, scale_param)
        k_val = 3.0
        k_name = "Mesokurtic (Normal Distribution)"
    elif dist_choice == 'laplace':
        y = stats.laplace.pdf(x, 0, scale_param / np.sqrt(2))
        k_val = 6.0
        k_name = "Leptokurtic (Laplace / Heavy-Tailed)"
    else:
        width = scale_param * np.sqrt(3)
        y = stats.uniform.pdf(x, -width, 2*width)
        k_val = 1.8
        k_name = "Platykurtic (Uniform / Light-Tailed)"
        
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(x, y, color='#2980b9', lw=2.5, label=f'{k_name}')
    ax.plot(x, stats.norm.pdf(x, 0, 1), color='grey', linestyle='--', alpha=0.6, label='Reference Standard Normal')
    
    ax.set_title(f'Interactive Kurtosis\nKurtosis ≈ {k_val:.1f} (Excess Kurtosis ≈ {k_val - 3:.1f})')
    ax.set_xlabel('Value')
    ax.set_ylabel('Probability Density')
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

interact(
    plot_interactive_kurtosis,
    dist_choice=Dropdown(options=[('Mesokurtic (Normal)', 'normal'), ('Leptokurtic (Laplace / Heavy Tails)', 'laplace'), ('Platykurtic (Uniform / Flat)', 'uniform')], value='normal', description='Kurtosis Type:'),
    scale_param=FloatSlider(min=0.5, max=3.0, step=0.1, value=1.0, description='Scale / Width:')
);


interactive(children=(Dropdown(description='Kurtosis Type:', options=(('Mesokurtic (Normal)', 'normal'), ('Lep…